**Technical Assessment: Weakly-Supervised Segmentation**

**Objective:** Implement a point-based segmentation pipeline using Partial Cross-Entropy Loss on remote sensing data.

**Task 1:** Implement Partial Cross Entropy Loss

**Task 2:** Dataset Preparation & Network Integration

**Task 3:** Experiments & Results

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
import random


**Implement Partial Cross Entropy Loss**

In [17]:
class PartialFocalCELoss(nn.Module):
    def __init__(self, ignore_index=-1, alpha=1.0, gamma=2.0):
        """
        Partial Focal Cross Entropy Loss for point-based segmentation.
        Matches the formula: pfCE = sum(Focal_Loss * Mask) / sum(Mask)
        """
        super(PartialFocalCELoss, self).__init__()
        self.ignore_index = ignore_index
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        # 1. Create a mask of the valid (labeled) pixels
        valid_mask = (targets != self.ignore_index).float()
        
        # 2. To avoid index errors out-of-bounds during raw CE calculation, 
        # replace ignore_index with a valid index (e.g., 0) in a temporary target tensor.
        # This works perfectly because the valid_mask will completely discard these positions.
        clean_targets = torch.where(targets == self.ignore_index, torch.zeros_like(targets), targets)
        
        # 3. Get raw, unreduced pixel-wise Cross Entropy Loss
        ce_loss = F.cross_entropy(inputs, clean_targets, reduction='none')
        
        # 4. Calculate pt (probability of the target class)
        pt = torch.exp(-ce_loss)
        
        # 5. Apply the Focal Loss modifier pixel-by-pixel
        focal_loss = self.alpha * ((1 - pt) ** self.gamma) * ce_loss
        
        # 6. Apply mask: Force ignored locations to have exactly 0 loss
        masked_focal_loss = focal_loss * valid_mask
        
        # 7. Sum the focal loss and divide by the number of valid pixels
        pfce_loss = masked_focal_loss.sum() / (valid_mask.sum() + 1e-8)
        
        return pfce_loss

In [4]:
import os

# 1. Install the Kaggle CLI tool
!pip install --quiet kaggle

# 2. Configure the global directory and move the pre-uploaded kaggle.json
print("Configuring Kaggle API credentials...")
!mkdir -p ~/.kaggle
!cp ./kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Clean up the file from your working directory for security
if os.path.exists("./kaggle.json"):
    os.remove("./kaggle.json")

# 3. Download and unzip the DeepGlobe dataset
print("Downloading DeepGlobe Land Cover dataset via Kaggle API...")
!kaggle datasets download -d balraj98/deepglobe-land-cover-classification-dataset

# Remove existing directory to ensure a clean extraction
!rm -rf ./deepglobe_data

# Unzip with -o to force overwrite and -q for quiet output
print("Extracting archive to local compute storage...")
!unzip -o -q deepglobe-land-cover-classification-dataset.zip -d ./deepglobe_data

# Clean up the zip file to save disk space on your Azure Compute instance
if os.path.exists("deepglobe-land-cover-classification-dataset.zip"):
    os.remove("deepglobe-land-cover-classification-dataset.zip")

print("Download and extraction complete.")


Configuring Kaggle API credentials...
cp: cannot stat './kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/balraj98/deepglobe-land-cover-classification-dataset
License(s): other
100%|██████████████████████████████████████| 2.74G/2.74G [02:34<00:00, 18.0MB/s]
100%|██████████████████████████████████████| 2.74G/2.74G [02:34<00:00, 19.0MB/s]
Extracting archive to local compute storage...
Download and extraction complete.


In [18]:
import os
import glob
import torch
import numpy as np
import pandas as pd
from PIL import Image
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

class DeepGlobePointDataset(Dataset):
    def __init__(self, root_dir, split='train', val_ratio=0.10, points_per_class=10, ignore_index=-1, transform=None):
        """
        Args:
            root_dir (str): Path to the extracted dataset (e.g., './deepglobe_data')
            split (str): 'train' or 'valid'
            points_per_class (int): Number of point labels to retain per class.
            ignore_index (int): The value to assign to unlabeled pixels.
            transform: PyTorch vision transforms.
        """
        self.root_dir = root_dir
        self.points_per_class = points_per_class
        self.ignore_index = ignore_index
        self.transform = transform
        
        # DeepGlobe maps files using a metadata CSV
        metadata_path = os.path.join(root_dir, 'metadata.csv')
        if not os.path.exists(metadata_path):
            raise FileNotFoundError(f"Metadata file not found at {metadata_path}. Check your root directory path.")
            
        df = pd.read_csv(metadata_path)
        df = df[df['split'] == 'train'].reset_index(drop=True)

        # Enforce a deterministic random split using a fixed seed (ensures train/val never mix)
        np.random.seed(42)
        total_samples = len(df)
        val_size = int(total_samples * val_ratio)
        shuffled_indices = np.random.permutation(total_samples)
        
        val_indices = shuffled_indices[:val_size]
        train_indices = shuffled_indices[val_size:]
        
        if split == 'train':
            selected_df = df.iloc[train_indices].reset_index(drop=True)
        elif split == 'valid':
            selected_df = df.iloc[val_indices].reset_index(drop=True)
        else:
            raise ValueError("Split must be either 'train' or 'valid'")

        # Map IDs to actual absolute image and mask paths in the flat directory
        self.image_paths = []
        self.mask_paths = []

        # Map filenames dynamically using the exact paths written inside the CSV fields
        for _, row in selected_df.iterrows():
            img_path = os.path.join(root_dir, row['sat_image_path'])
            mask_path = os.path.join(root_dir, row['mask_path'])
            
            if os.path.exists(img_path):
                self.image_paths.append(img_path)
                self.mask_paths.append(mask_path)
            else:
                # Fallback check in case files were extracted directly into a flat root folder
                flat_img = os.path.join(root_dir, os.path.basename(row['sat_image_path']))
                flat_mask = os.path.join(root_dir, os.path.basename(row['mask_path']))
                if os.path.exists(flat_img):
                    self.image_paths.append(flat_img)
                    self.mask_paths.append(flat_mask)
                else:
                    raise FileNotFoundError(f"Could not locate {row['sat_image_path']} anywhere under {root_dir}")

        # DeepGlobe RGB to Class mapping
        self.color_map = {
            (0, 255, 255): 0,    # Urban
            (255, 255, 0): 1,    # Agriculture
            (255, 0, 255): 2,    # Rangeland
            (0, 255, 0): 3,      # Forest
            (0, 0, 255): 4,      # Water
            (255, 255, 255): 5,  # Barren
            (0, 0, 0): 6         # Unknown
        }
        
        # Build a fast lookup table array for fast color translation
        self.lookup_table = np.zeros((256, 256, 256), dtype=np.int64)
        for rgb, idx in self.color_map.items():
            self.lookup_table[rgb[0], rgb[1], rgb[2]] = idx

    def _rgb_to_mask(self, mask_img):
        """Ultra-fast vector conversion using a 3D lookup array."""
        mask_np = np.array(mask_img)
        # Directly index the lookup table using channels
        mask_idx = self.lookup_table[mask_np[:, :, 0], mask_np[:, :, 1], mask_np[:, :, 2]]
        return torch.tensor(mask_idx, dtype=torch.long)

    def _simulate_points(self, mask):
        """Converts a dense mask into a weakly-supervised point mask."""
        simulated_mask = torch.full_like(mask, self.ignore_index)
        unique_classes = torch.unique(mask)
        
        for c in unique_classes:
            if c == self.ignore_index:
                continue
                
            coords = (mask == c).nonzero(as_tuple=False)
            if len(coords) == 0:
                continue
                
            num_points = min(self.points_per_class, len(coords))
            random_indices = torch.randperm(len(coords))[:num_points]
            sampled_coords = coords[random_indices]
            
            # Optimized non-loop assignment via tensor indices
            simulated_mask[sampled_coords[:, 0], sampled_coords[:, 1]] = c
            
        return simulated_mask

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]
        
        image = Image.open(img_path).convert("RGB")
        mask_rgb = Image.open(mask_path).convert("RGB")

        # 1. Convert RGB to dense class integers at native resolution
        dense_mask = self._rgb_to_mask(mask_rgb)
    
    # 2. Resize masks to 512x512 using NEAREST interpolation BEFORE point simulation
    # This prevents point annotations from accidentally being dropped during downsampling
        dense_mask = TF.resize(dense_mask.unsqueeze(0), size=[512, 512], 
                           interpolation=TF.InterpolationMode.NEAREST).squeeze(0)
    
    # 3. Apply the Task 2 point simulation on the scaled mask
        point_mask = self._simulate_points(dense_mask)
        
        if self.transform:
            image = self.transform(image)
        
        return image, point_mask, dense_mask

print("Task 2 Setup: DeepGlobe Dataset class with point simulation ready.")

Task 2 Setup: DeepGlobe Dataset class with point simulation ready.


In [23]:
import torchvision.transforms as T
from torch.utils.data import DataLoader

transform = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Query hardware state and set global variable context
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Overwrite dataset configurations entirely
dataset = DeepGlobePointDataset(root_dir='./deepglobe_data', split='train', points_per_class=20, transform=transform)

# Setting num_workers=0 clears Azure compute background multi-process deadlocks
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)

print(f"Data pipeline bound. Confirmed safe path index 0 location: {dataset.image_paths[0]}")


Data pipeline bound. Confirmed safe path index 0 location: ./deepglobe_data/train/332354_sat.jpg


In [24]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

# 1. Hyperparameters
BATCH_SIZE = 4
NUM_CLASSES = 7  # Based on our DeepGlobe color map
LEARNING_RATE = 1e-4
EPOCHS = 5

# Define a professional transformation pipeline for satellite imagery
# This avoids OOM crashes on your Azure GPU and aligns with ImageNet features
transform = T.Compose([
    T.Resize((512, 512)),  # Scales down 2448x2448 images to fit in GPU memory
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Matches pretrained backbone expectations
])

# 2. Initialize the Dataset and DataLoader
# We use the fast metadata-driven class we configured in the previous step
dataset = DeepGlobePointDataset(root_dir='./deepglobe_data', split='train', points_per_class=50, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

# 3. Load Pre-trained Model with Explicit Auxiliary Loss Deactivation
weights = DeepLabV3_ResNet50_Weights.DEFAULT
# Setting aux_loss=False stops the model from expecting a 21-class auxiliary output
model = deeplabv3_resnet50(weights=weights, aux_loss=True)

# Re-route the main classifier layer to output 7 channels
model.classifier[4] = nn.Conv2d(256, NUM_CLASSES, kernel_size=(1, 1), stride=(1, 1))

# Modify the auxiliary classifier layer to output 7 classes as well
model.aux_classifier[4] = nn.Conv2d(256, NUM_CLASSES, kernel_size=(1, 1), stride=(1, 1))

model = model.to(device)

# 4. Initialize our bulletproof custom Partial CE Loss and the Optimizer
# Ensure alpha and gamma match your assessment criteria
criterion = PartialFocalCELoss(ignore_index=-1, alpha=1.0, gamma=2.0)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("Task 2 Model Setup: DeepLabV3 successfully initialized without auxiliary conflicts.")
print(f"Data pipeline configured. Expected input batch dimensions: [{BATCH_SIZE}, 3, 512, 512]")


Task 2 Model Setup: DeepLabV3 successfully initialized without auxiliary conflicts.
Data pipeline configured. Expected input batch dimensions: [4, 3, 512, 512]


In [25]:
print("Starting Training...")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for i, (images, point_masks, dense_masks) in enumerate(dataloader):
        
        # Move data to GPU
        images = images.to(device)
        point_masks = point_masks.to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        # DeepLabV3 returns an OrderedDict. The actual output is under the 'out' key.
        outputs = model(images)['out']

        # Calculate our custom Partial Cross Entropy Loss
        # outputs shape: [B, 7, 512, 512], point_masks shape: [B, 512, 512]
        loss = criterion(outputs, point_masks)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if (i + 1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}], Step [{i+1}/{len(dataloader)}], Loss: {loss.item():.4f}")

    print(f"--- Epoch {epoch+1} Average Loss: {running_loss/len(dataloader):.4f} ---")

print("Training Complete!")

Starting Training...
Epoch [1/5], Step [5/180], Loss: 1.3542
Epoch [1/5], Step [10/180], Loss: 1.3639
Epoch [1/5], Step [15/180], Loss: 1.4014
Epoch [1/5], Step [20/180], Loss: 1.1129
Epoch [1/5], Step [25/180], Loss: 1.1972
Epoch [1/5], Step [30/180], Loss: 1.0893
Epoch [1/5], Step [35/180], Loss: 1.0876
Epoch [1/5], Step [40/180], Loss: 0.9580
Epoch [1/5], Step [45/180], Loss: 0.9364
Epoch [1/5], Step [50/180], Loss: 1.1606
Epoch [1/5], Step [55/180], Loss: 1.0081
Epoch [1/5], Step [60/180], Loss: 0.9884
Epoch [1/5], Step [65/180], Loss: 0.9746
Epoch [1/5], Step [70/180], Loss: 0.9042
Epoch [1/5], Step [75/180], Loss: 0.9767
Epoch [1/5], Step [80/180], Loss: 1.0930
Epoch [1/5], Step [85/180], Loss: 0.9548
Epoch [1/5], Step [90/180], Loss: 0.9906
Epoch [1/5], Step [95/180], Loss: 0.6323
Epoch [1/5], Step [100/180], Loss: 0.9296
Epoch [1/5], Step [105/180], Loss: 0.7304
Epoch [1/5], Step [110/180], Loss: 0.7850
Epoch [1/5], Step [115/180], Loss: 0.6517
Epoch [1/5], Step [120/180], Loss

In [26]:
import torch
import numpy as np

def evaluate_model_miou(model, val_dataloader, num_classes=7, device='cuda'):
    """
    Evaluates the model on the full dense masks of the validation set 
    and returns the Mean Intersection over Union (mIoU).
    """
    model.eval()
    
    # Initialize trackers for True Positives, False Positives, and False Negatives per class
    total_tp = torch.zeros(num_classes, device=device)
    total_fp = torch.zeros(num_classes, device=device)
    total_fn = torch.zeros(num_classes, device=device)
    
    print("Evaluating on Validation Set...")
    with torch.no_grad():
        for images, _, dense_masks in val_dataloader:
            images = images.to(device)
            dense_masks = dense_masks.to(device) # Shape: [B, 512, 512]
            
            # Forward pass
            outputs = model(images)['out'] # Shape: [B, 7, 512, 512]
            preds = torch.argmax(outputs, dim=1) # Shape: [B, 512, 512]
            
            # Calculate IoU elements for each class
            for c in range(num_classes):
                pred_mask = (preds == c)
                target_mask = (dense_masks == c)
                
                total_tp[c] += (pred_mask & target_mask).sum()
                total_fp[c] += (pred_mask & ~target_mask).sum()
                total_fn[c] += (~pred_mask & target_mask).sum()
                
    # Compute IoU per class: TP / (TP + FP + FN)
    iou_per_class = total_tp / (total_tp + total_fp + total_fn + 1e-8)
    
    # Calculate Mean IoU across all classes (excluding class 6 if 'Unknown' shouldn't count)
    valid_classes_iou = iou_per_class[:6] # DeepGlobe class 6 is 'Unknown/Void'
    miou = valid_classes_iou.mean().item()
    
    print(f"Validation mIoU: {miou:.4f}")
    for c, score in enumerate(iou_per_class.cpu().numpy()):
        print(f"  Class {c} IoU: {score:.4f}")
        
    return miou


In [27]:
# Create the validation dataset and dataloader once
val_transform = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_dataset = DeepGlobePointDataset(root_dir='./deepglobe_data', split='valid', transform=val_transform)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Validation Dataloader bound to valid ground-truth sub-split: {len(val_dataloader)} batches available.")

Validation Dataloader bound to valid ground-truth sub-split: 20 batches available.


In [28]:
miou_50 = evaluate_model_miou(model, val_dataloader, device=device)

Evaluating on Validation Set...
Validation mIoU: 0.5225
  Class 0 IoU: 0.6366
  Class 1 IoU: 0.7366
  Class 2 IoU: 0.2789
  Class 3 IoU: 0.6401
  Class 4 IoU: 0.4871
  Class 5 IoU: 0.3558
  Class 6 IoU: 0.0001
